# Window Functions: More Powerful Than groupBy

Window functions perform calculations **across a set of rows that are related to the current row**,
without collapsing those rows into a single output row the way `groupBy().agg()` does.

---

## The Core Difference

```
groupBy().agg()                      Window Function
──────────────────────               ──────────────────────────────────────
dept  salary                         dept  name    salary  dept_avg_salary
────  ──────      ──►                ────  ──────  ──────  ───────────────
Eng   90000       avg per dept       Eng   Alice   90000   85000
Eng   80000       ──►  Eng  85000    Eng   Bob     80000   85000
HR    60000            HR   60000    HR    Carol   60000   60000

Two rows became ONE.                 Three rows stayed THREE.
                                     Each row knows its group's avg.
```

This property makes window functions essential for:
- **Rankings** within groups (who is the top earner per department?)
- **Lead/Lag comparisons** (how did my salary change from last year?)
- **Running/rolling aggregates** (cumulative revenue, 7-day moving average)
- **Percentile bucketing** (which quartile does this salary fall into?)

---

## Notebook Roadmap

| Section | Topic |
|---------|-------|
| 1 | Setup & employee dataset |
| 2 | Window spec anatomy (theory) |
| 3 | Ranking functions |
| 4 | Lead & Lag |
| 5 | Rolling aggregates |
| 6 | Running totals |
| 7 | `ntile()` for quartile buckets |


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("Window Functions")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Employee dataset: department, name, salary, hire_date.
# Multiple employees per department enables meaningful partition-by-dept windows.
employee_data = [
    ("Engineering", "Alice",   95000, "2019-03-15"),
    ("Engineering", "Bob",     82000, "2020-07-01"),
    ("Engineering", "Charlie", 95000, "2018-11-20"),  # same salary as Alice → tie in ranking
    ("Engineering", "Diana",   110000, "2017-06-30"),
    ("Engineering", "Eve",     76000, "2022-01-10"),
    ("Marketing",   "Frank",   68000, "2021-05-05"),
    ("Marketing",   "Grace",   72000, "2020-08-15"),
    ("Marketing",   "Hank",    68000, "2019-12-01"),  # tie with Frank
    ("Marketing",   "Irene",   85000, "2018-04-20"),
    ("HR",          "Jake",    55000, "2023-02-01"),
    ("HR",          "Kate",    62000, "2021-09-10"),
    ("HR",          "Leo",     58000, "2022-06-15"),
]

schema = StructType([
    StructField("dept",      StringType(),  False),
    StructField("name",      StringType(),  False),
    StructField("salary",    IntegerType(), False),
    StructField("hire_date", StringType(),  True),
])

emp_df = (
    spark.createDataFrame(employee_data, schema=schema)
    .withColumn("hire_date", F.to_date(F.col("hire_date"), "yyyy-MM-dd"))  # proper DateType
)

emp_df.printSchema()
emp_df.orderBy("dept", "salary").show()

## Window Spec Anatomy

A `WindowSpec` is the configuration object that tells Spark **which rows** form the "window" for
each input row. You build one using `Window.partitionBy().orderBy().rowsBetween()` / `rangeBetween()`.

```
w = (
    Window
    .partitionBy("dept")                          # (1) group rows by department
    .orderBy(F.col("salary").desc())              # (2) sort within each partition
    .rowsBetween(Window.unboundedPreceding, 0)    # (3) frame: all rows up to current
)
```

### The Three Clauses

| Clause | Purpose | Required? |
|--------|---------|----------|
| `partitionBy(col)` | Defines independent groups (like GROUP BY) | No — omitting means the whole DataFrame is one window |
| `orderBy(col)` | Determines row ordering within each partition | Required for ranking, lead/lag, cumulative functions |
| `rowsBetween(start, end)` / `rangeBetween(start, end)` | Frame boundary — which rows feed the aggregate | Optional; default depends on function |

### Frame Boundary Constants

```
Window.unboundedPreceding  →  the very first row of the partition
Window.currentRow          →  the row being computed (position 0)
Window.unboundedFollowing  →  the very last row of the partition
-2                         →  2 rows BEFORE the current row
+1                         →  1 row AFTER the current row
```

### ROWS BETWEEN vs RANGE BETWEEN

- **ROWS BETWEEN** counts physical rows — tie-breaking is positional.
- **RANGE BETWEEN** counts logical value offsets — all rows with the same ORDER BY value are treated
  as equal and all fall inside the frame together. Useful for time-series (e.g., all events within
  the last 7 days) but can cause surprising behavior with ties.

### Default Frames (when no frame is specified)

```
No orderBy  →  ROWS BETWEEN unboundedPreceding AND unboundedFollowing  (whole partition)
With orderBy → RANGE BETWEEN unboundedPreceding AND currentRow         (running aggregate)
```


In [ ]:
# ============================================================
# SECTION 3: Ranking Functions
# ============================================================
# row_number(), rank(), dense_rank(), percent_rank() all require
# an ORDER BY in the window spec — they are inherently order-dependent.
#
# The key difference is how TIES are handled:
#
#  salary  row_number  rank  dense_rank  percent_rank
#  110000       1        1        1         0.00
#   95000       2        2        2         0.25   ← tie
#   95000       3        2        2         0.25   ← tie (rank stays 2, dense stays 2)
#   82000       4        4        3         0.75   ← rank SKIPS to 4, dense_rank goes to 3
#   76000       5        5        4         1.00

# Window: partition by department, highest salary first.
rank_window = Window.partitionBy("dept").orderBy(F.col("salary").desc())

ranked_df = emp_df.select(
    "dept",
    "name",
    "salary",
    # row_number: unique sequential number — no gaps, no tie awareness
    F.row_number().over(rank_window).alias("row_num"),
    # rank: tied rows get the same rank, then rank SKIPS (1,2,2,4)
    F.rank().over(rank_window).alias("rank"),
    # dense_rank: tied rows get the same rank, NO skipping (1,2,2,3)
    F.dense_rank().over(rank_window).alias("dense_rank"),
    # percent_rank: (rank-1)/(n-1) — produces a 0.0 to 1.0 score
    F.round(F.percent_rank().over(rank_window), 3).alias("pct_rank"),
)

ranked_df.orderBy("dept", F.col("salary").desc()).show()

# Common pattern: get only the TOP-1 employee per department (like GROUP BY with argmax).
print("=== Top earner per department ===")
ranked_df.filter(F.col("row_num") == 1).select("dept", "name", "salary").show()

In [ ]:
# ============================================================
# SECTION 4: Lead and Lag
# ============================================================
# lag(col, offset, default)  → look BACKWARD (previous rows)
# lead(col, offset, default) → look FORWARD  (next rows)
#
# Classic use cases:
#  - Compare employee salary to the one just above/below them in the pay scale
#  - Calculate period-over-period change in a time series
#  - Detect state transitions (previous status != current status)

# Sort by hire_date ASC within each department to build a chronological view.
time_window = Window.partitionBy("dept").orderBy("hire_date")

lead_lag_df = emp_df.select(
    "dept",
    "name",
    "hire_date",
    "salary",
    # Previous hire's salary in the same department (1 row back).
    # Third argument is the default value when no previous row exists.
    F.lag("salary", 1, None).over(time_window).alias("prev_salary"),
    # Next hire's salary in the same department (1 row forward).
    F.lead("salary", 1, None).over(time_window).alias("next_salary"),
)

# Compute the salary delta vs the previous hire.
lead_lag_df = lead_lag_df.withColumn(
    "salary_vs_prev",
    F.col("salary") - F.col("prev_salary")  # null for the first hire in each dept
)

lead_lag_df.orderBy("dept", "hire_date").show()

# You can also use lag with offset=2 to look two rows back, or
# lead("hire_date", 1) to compute days between consecutive hires.
print("=== Days between consecutive hires (per dept) ===")
emp_df.select(
    "dept", "name", "hire_date",
    F.lead("hire_date", 1).over(time_window).alias("next_hire_date")
).withColumn(
    "days_to_next_hire",
    F.datediff(F.col("next_hire_date"), F.col("hire_date"))
).orderBy("dept", "hire_date").show()

In [ ]:
# ============================================================
# SECTION 5: Rolling Aggregates (Sliding Window)
# ============================================================
# A rolling (or sliding) window considers a FIXED number of preceding rows.
# Classic example: 3-day moving average of daily revenue.
#
# Frame:  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
#         means: current row + the 2 rows immediately before it = 3 rows total.

# Sort employees by salary ascending within each department.
salary_window = Window.partitionBy("dept").orderBy("salary")

# 3-row rolling average — useful for smoothing out individual outliers.
rolling_window = salary_window.rowsBetween(-2, Window.currentRow)

rolling_df = emp_df.select(
    "dept",
    "name",
    "salary",
    # 3-row rolling average: avg of (current row + up to 2 rows before)
    F.round(
        F.avg("salary").over(rolling_window), 0
    ).alias("rolling_3_avg"),
    # 3-row rolling min and max
    F.min("salary").over(rolling_window).alias("rolling_3_min"),
    F.max("salary").over(rolling_window).alias("rolling_3_max"),
)

print("=== 3-row rolling statistics (ordered by salary asc within dept) ===")
rolling_df.orderBy("dept", "salary").show()

# Notice: the FIRST row in each partition uses only itself (1-row window).
# The SECOND row uses 2 rows. The THIRD and beyond use 3 rows.
# This "ramp-up" behaviour is inherent to ROWS BETWEEN with PRECEDING.

In [ ]:
# ============================================================
# SECTION 6: Running Totals (Cumulative Sum)
# ============================================================
# A running total accumulates from the FIRST row of the partition up to
# the CURRENT row.
#
# Frame:  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
#
# This is actually the DEFAULT frame when orderBy() is specified, so
# you can omit the explicit rowsBetween — but being explicit is clearer.

cumulative_window = (
    Window
    .partitionBy("dept")
    .orderBy("hire_date")                              # oldest hire first
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

cumulative_df = emp_df.select(
    "dept",
    "name",
    "hire_date",
    "salary",
    # Running total salary cost as employees are hired over time.
    F.sum("salary").over(cumulative_window).alias("cumulative_payroll"),
    # Running count of employees hired so far in this department.
    F.count("name").over(cumulative_window).alias("headcount_so_far"),
    # Running average salary (cumulative average)
    F.round(
        F.avg("salary").over(cumulative_window), 0
    ).alias("cumulative_avg_salary"),
)

print("=== Running totals as employees are hired (oldest first per dept) ===")
cumulative_df.orderBy("dept", "hire_date").show()

# For a WHOLE-PARTITION total in a window (not cumulative), use:
#   Window.partitionBy("dept")  ← no orderBy, no frame
# This broadcasts the total to every row without collapsing them.
whole_window = Window.partitionBy("dept")
emp_df.withColumn(
    "dept_total_payroll", F.sum("salary").over(whole_window)
).withColumn(
    "pct_of_dept_payroll",
    F.round(F.col("salary") / F.col("dept_total_payroll") * 100, 1)
).orderBy("dept", "salary").show()

In [ ]:
# ============================================================
# SECTION 7: ntile() — Salary Quartiles Within Each Department
# ============================================================
# ntile(n) divides the partition into n buckets of approximately equal size
# and assigns each row a bucket number from 1 to n.
#
# ntile(4) = quartiles:
#   bucket 1 = bottom 25%  (Q1)
#   bucket 2 = 25-50%      (Q2)
#   bucket 3 = 50-75%      (Q3)
#   bucket 4 = top 25%     (Q4)
#
# When partition size isn't divisible by n, the FIRST buckets get one
# extra row (Spark uses the larger-first allocation).

quartile_window = Window.partitionBy("dept").orderBy(F.col("salary").asc())

quartile_df = emp_df.select(
    "dept",
    "name",
    "salary",
    F.ntile(4).over(quartile_window).alias("salary_quartile"),  # Q1..Q4
    F.ntile(10).over(quartile_window).alias("salary_decile"),   # D1..D10
)

print("=== Salary quartiles and deciles within each department ===")
quartile_df.orderBy("dept", "salary").show()

# Summarise: average salary per quartile per department
print("=== Mean salary per quartile per department ===")
quartile_df.groupBy("dept", "salary_quartile").agg(
    F.avg("salary").alias("avg_salary"),
    F.count("name").alias("headcount")
).orderBy("dept", "salary_quartile").show()

In [ ]:
# Clean shutdown.
spark.stop()
print("SparkSession stopped.")